# Debate & Critique Pattern

*Notebook 17*

Improve output quality by having agents challenge and refine each other's work.
<br>
<br>
**Topics:**

- Why critique improves quality

- Proposer / Critic architecture

- Passing output between agents with `result.final_output`

- Critique loop termination rules — avoiding infinite loops

- Improving quality through internal debate

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner

# Notebook-specific imports
import asyncio

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

print("✅ Ready!")

---

## 🎯 The Problem

A single agent produces its first reasonable answer and stops — it has no incentive to challenge its own assumptions. Separating the roles of proposer and critic produces outputs that have been challenged and improved rather than just accepted.

---

## 💡 Part 1: The Pattern

The Debate & Critique pattern uses sequential `Runner.run()` calls, passing output forward at each step:

```
Topic
  ↓
Proposer   →  initial response
  ↓
Critic      →  feedback and weaknesses
  ↓
Proposer   →  revised response (using the critique)
  ↓
Final output
```

Each agent gets the previous agent's output as part of its input. No special SDK features required — just sequential `Runner.run()` calls with `result.final_output` passed forward.

---

## ✍️ Part 2: Basic Proposer / Critic

We'll start with a simple writing improvement pipeline — a writer proposes, an editor critiques, and the writer revises.

In [ ]:
writer_instructions = (
    "You write clear, concise explanations of technical concepts.\n"
    "Target audience: beginner developers. Use plain language and one concrete example."
)

writer_agent = Agent(
    name="Writer",
    instructions=writer_instructions,
    model=MODEL
)

editor_instructions = (
    "You are a critical technical editor. Review the given explanation and identify:\n"
    "1. Any unclear or jargon-heavy language\n"
    "2. Missing or weak examples\n"
    "3. Logical gaps or oversimplifications\n"
    "4. What specifically should be improved\n"
    "Be direct and specific. Do not rewrite — only critique."
)

editor_agent = Agent(
    name="Editor",
    instructions=editor_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Writer and Editor agents ready")

In [ ]:
topic = "Explain what an API is"

print("✍️ STEP 1: FIRST DRAFT")
print("=" * 60)

draft_result = await Runner.run(writer_agent, input=topic)

first_draft = draft_result.final_output

print(first_draft)
print("=" * 60)

In [ ]:
print("🔍 STEP 2: CRITIQUE")
print("=" * 60)

critique_input = f"Please critique this explanation:\n\n{first_draft}"

critique_result = await Runner.run(editor_agent, input=critique_input)

critique = critique_result.final_output

print(critique)
print("=" * 60)

In [ ]:
print("✅ STEP 3: REVISED DRAFT")
print("=" * 60)

revision_input = (
    f"Here is your original explanation:\n\n{first_draft}\n\n"
    f"Here is the editor's critique:\n\n{critique}\n\n"
    f"Please rewrite the explanation addressing all the feedback."
)

revision_result = await Runner.run(writer_agent, input=revision_input)

revised_draft = revision_result.final_output

print(revised_draft)
print("=" * 60)

### 💡 Why This Works

The critic has one job — find weaknesses — so it's more likely to surface them than an agent trying to do everything at once. The writer then has specific, targeted feedback to act on rather than vague instructions to "improve."

---

## 🔄 Part 3: The Critique Loop — With Termination Rules

A single critique cycle helps. A loop with a quality gate helps more — keep refining until the output meets a minimum standard, or until a maximum iteration count is reached.

⚠️ Always set a `max_iterations` limit. Without it, a loop can run indefinitely and rack up unexpected API costs.

Judging quality is a reasoning task — we use `REASONING_MODEL` here because the judge needs to evaluate the explanation carefully, while the writer can stay on the cheaper `MODEL`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Annotated

class QualityScore(BaseModel):
    score: Annotated[int, Field(ge=1, le=10)]

judge_instructions = (
    "You evaluate explanation quality on a scale of 1-10.\n"
    "Criteria:\n"
    "- Clarity (is it easy to understand?)\n"
    "- Accuracy (is it technically correct?)\n"
    "- Example quality (is the example concrete and helpful?)\n\n"
    "Evaluate the explanation and return your score."
)

quality_judge_agent = Agent(
    name="QualityJudge",
    instructions=judge_instructions,
    model=REASONING_MODEL,
    output_type=QualityScore
)

# --------------------------------------------------------------
print("✅ QualityJudge agent ready")

In [ ]:
async def critique_loop(topic, quality_threshold=8, max_iterations=3):
    """Run proposer/critic loop until quality threshold is met."""

    print(f"🔄 CRITIQUE LOOP: {topic}")
    print(f"    Target quality: {quality_threshold}/10 | Max iterations: {max_iterations}")
    print("=" * 60)

    # -------------------------------------------------------
    # Initial draft
    # -------------------------------------------------------

    draft_result = await Runner.run(writer_agent, input=topic)

    current_draft = draft_result.final_output
    print(f"\n📝 Initial draft ({len(current_draft)} chars)")

    tripped = False
    for iteration in range(1, max_iterations + 1):

        # -------------------------------------------------------
        # Score current draft
        # -------------------------------------------------------

        score_result = await Runner.run(quality_judge_agent, input=f"Rate this explanation:\n\n{current_draft}")

        score = score_result.final_output.score

        print(f"\n🔄 Iteration {iteration}: Quality score = {score}/10")

        if score >= quality_threshold:
            print(f"✅ Quality threshold met — stopping at iteration {iteration}")
            tripped = True
            break

        # -------------------------------------------------------
        # Critique and revise
        # -------------------------------------------------------

        critique_result = await Runner.run(editor_agent, input=f"Critique this explanation:\n\n{current_draft}")

        critique = critique_result.final_output

        revision_result = await Runner.run(writer_agent, input=f"Original:\n{current_draft}\n\nCritique:\n{critique}\n\nRewrite addressing all feedback.")

        current_draft = revision_result.final_output
        print(f"    Revised to {len(current_draft)} chars")
    if not tripped:
        print(f"\n⚠️ Max iterations reached — returning the most recent revision. Treat `max_iterations` as a cost cap, not a quality guarantee. Consider raising iterations or lowering the threshold.")

    print("\n" + "=" * 60)
    print("📋 FINAL OUTPUT:")
    print(current_draft)
    return current_draft


# --------------------------------------------------------------
print("✅ critique_loop() ready")

#### Run the Loop

In [ ]:
final_output = await critique_loop(
    topic="Explain what recursion is in programming",
    quality_threshold=8,
    max_iterations=3
)

---

## 🥊 Part 4: True Debate — Two Opposing Positions

Use the critique loop to refine a single output toward a quality bar; use a true debate when the question has legitimately opposing positions and you want both surfaced before deciding.

Both agents receive the same topic and independently argue their positions. Then the advocate rebuts the skeptic's argument. A synthesizer draws a balanced conclusion from all three rounds.

1. Advocate → argues for

2. Skeptic → argues against (independent, same prompt)

3. Advocate → rebuts the skeptic

4. Synthesizer → balanced conclusion


In [ ]:
advocate_agent = Agent(
    name="Advocate",
    instructions="You argue in strong favor of the given position. Make the best case possible. Be specific.",
    model=MODEL
)

skeptic_agent = Agent(
    name="Skeptic",
    instructions="You argue strongly against the given position. Make the strongest opposing case possible. Be specific.",
    model=MODEL
)

synthesis_agent = Agent(
    name="Synthesizer",
    instructions="You synthesize a debate into a balanced conclusion. Extract the strongest points from both sides and produce a nuanced final position.",
    model=REASONING_MODEL
)

# --------------------------------------------------------------
print("✅ Debate agents ready")

#### Run the Debate

In [ ]:
debate_topic = "Microservices are better than monoliths for most software teams"

print("🥊 DEBATE")
print("=" * 60)
print(f"Topic: {debate_topic}\n")

# Round 1: Independent positions — run in parallel since they don't depend on each other
print("Round 1: Independent positions\n")

advocate_result, skeptic_result = await asyncio.gather(

    Runner.run(advocate_agent, input=f"Argue in favor of: {debate_topic}"),

    Runner.run(skeptic_agent, input=f"Argue against: {debate_topic}")

)

advocate_position = advocate_result.final_output
print(f"📢 Advocate:\n{advocate_position}\n")

skeptic_position = skeptic_result.final_output
print(f"🔍 Skeptic:\n{skeptic_position}\n")

# Round 2: Rebuttal
print("Round 2: Rebuttal\n")

rebuttal_input = (
    f"You previously argued: {advocate_position}\n\n"
    f"The skeptic responded: {skeptic_position}\n\n"
    f"Rebut the skeptic's strongest points."
)

rebuttal_result = await Runner.run(advocate_agent, input=rebuttal_input)

rebuttal = rebuttal_result.final_output
print(f"📢 Advocate rebuttal:\n{rebuttal}\n")

# Round 3: Synthesis
print("Round 3: Synthesis\n")

synthesis_input = (
    f"Synthesize this debate into a balanced conclusion.\n\n"
    f"Topic: {debate_topic}\n\n"
    f"Advocate's position:\n{advocate_position}\n\n"
    f"Skeptic's position:\n{skeptic_position}\n\n"
    f"Advocate's rebuttal:\n{rebuttal}"
)

synthesis_result = await Runner.run(synthesis_agent, input=synthesis_input)

print("⚖️ SYNTHESIS")
print("=" * 60)
print(synthesis_result.final_output)
print("=" * 60)


### 💡 Why This Works

Both agents receive the same topic and argue independently — the advocate finds the strongest case for, the skeptic finds the strongest case against. The rebuttal step forces the advocate to engage with real counterarguments rather than ignore them. The synthesizer then has a richer set of positions to draw from than a simple argument + critique would produce.

Parallelism here is an optimization carried over from Lesson 16 — the advocate and skeptic argue independently, so we run them with `gather`. The debate pattern itself is about opposing roles and synthesis, not parallelism.

---

## 💪 Practice Exercises

### Exercise 1: Code Review Pipeline

*Covers: debate pattern — Proposer/Critic loop*

Build a proposer/critic pipeline where one agent writes code and another reviews it for bugs and improvements.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Code Review Pipeline
# --------------------------------------------------------------
# Objective: Writer writes code, reviewer critiques, writer revises.

code_task = "Write a Python function that finds duplicate values in a list"

# TODO 1: Create a CodeWriter agent with instructions to write clean,
#           well-commented Python functions

# TODO 2: Create a CodeReviewer agent with instructions to check for:
#           - edge cases not handled
#           - performance issues
#           - missing docstring or comments
#           Critic only — do not rewrite

# TODO 3: Run the pipeline:
#           a) CodeWriter writes the function
#           b) CodeReviewer critiques it
#           c) CodeWriter revises based on feedback
#           Print each step with a clear header

# --- Write your code below this line ---

### Exercise 2: Business Plan Debate

*Covers: debate pattern — multi-angle critique*

Use the debate pattern to evaluate a business idea from multiple angles.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Business Plan Debate
# --------------------------------------------------------------
# Objective: Apply the full debate pattern to a business idea.

business_idea = "A subscription service that delivers curated book summaries weekly"

# TODO 1: Create a BusinessAdvocate agent — argues strongly in favor of the idea

# TODO 2: Create a BusinessSkeptic agent — argues strongly against the idea

# TODO 3: Create a BusinessAnalyst agent — synthesizes all sides into an investment verdict

# TODO 4: Run advocate and skeptic independently on the same business_idea prompt

# TODO 5: Run advocate again with the skeptic's argument to generate a rebuttal

# TODO 6: Run the analyst with all three outputs — advocate position,
#           skeptic position, and rebuttal — to produce a final verdict
#           Print each stage clearly

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Separation of roles improves quality:**

- A dedicated critic finds more weaknesses than a generalist trying to self-improve

- Each agent does one job well rather than many jobs adequately
<br>
<br>
**Loops need termination rules:**

- Always set `max_iterations` — a loop without a limit will run until you run out of tokens or money

- A quality threshold lets you stop early when output is good enough
<br>
<br>
**Output chaining is the mechanism:**

- `result.final_output` from one agent becomes part of the next agent's input

- No special SDK features needed — just sequential `Runner.run()` calls
<br>
<br>
**Independent positions surface stronger tradeoffs:**

- Running advocate and skeptic independently on the same topic produces richer arguments than sequential critique

- A synthesizer drawing from both positions produces a more balanced conclusion than either agent alone

---

## 📍 Next Step

**Notebook 18: Capstone #2 — Multi-Agent Research Team**  

Combine handoffs, agents-as-tools, parallel execution, and critique into a full multi-agent research pipeline.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-17-debate-and-critique)

---